### Tugas Mandiri 3 
**Nama :** Putri Senja Nuril Azizah  
**NPM  :** 2505060036

#### A. Membangun Struktur Direktori HDFS

In [8]:
!hdfs dfs -mkdir -p /user/putrisenja/ecommerce/raw
!hdfs dfs -mkdir -p /user/putrisenja/ecommerce/processed
!hdfs dfs -ls /user/putrisenja/ecommerce

Found 2 items
drwxr-xr-x   - putrisenja supergroup          0 2026-09-09 06:38 /user/putrisenja/ecommerce/processed
drwxr-xr-x   - putrisenja supergroup          0 2026-09-09 06:42 /user/putrisenja/ecommerce/raw


#### B. Mengunggah Data Mentah ke HDFS

In [9]:
!hdfs dfs -put transaksi_magelang.csv transaksi_yogyakarta.csv transaksi_semarang.csv /user/putrisenja/ecommerce/raw
!hdfs dfs -ls /user/putrisenja/ecommerce/raw

put: `/user/putrisenja/ecommerce/raw/transaksi_magelang.csv': File exists
put: `/user/putrisenja/ecommerce/raw/transaksi_yogyakarta.csv': File exists
put: `/user/putrisenja/ecommerce/raw/transaksi_semarang.csv': File exists
Found 3 items
-rw-r--r--   1 putrisenja supergroup      12329 2026-09-09 06:42 /user/putrisenja/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 putrisenja supergroup      12154 2026-09-09 06:42 /user/putrisenja/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 putrisenja supergroup      12684 2026-09-09 06:42 /user/putrisenja/ecommerce/raw/transaksi_yogyakarta.csv


#### C. Membaca Kembali dan Menggabungkan Data dari HDFS

In [10]:
# Mengunduh ketiga berkas
!hdfs dfs -get /user/putrisenja/ecommerce/raw/transaksi_magelang.csv magelang_dari_hdfs.csv
!hdfs dfs -get /user/putrisenja/ecommerce/raw/transaksi_semarang.csv semarang_dari_hdfs.csv
!hdfs dfs -get /user/putrisenja/ecommerce/raw/transaksi_yogyakarta.csv yogyakarta_dari_hdfs.csv

get: `magelang_dari_hdfs.csv': File exists
get: `semarang_dari_hdfs.csv': File exists
get: `yogyakarta_dari_hdfs.csv': File exists


In [11]:
import pandas as pd 
df_magelang = pd.read_csv("magelang_dari_hdfs.csv")
df_semarang = pd.read_csv("semarang_dari_hdfs.csv")
df_yogyakarta = pd.read_csv("yogyakarta_dari_hdfs.csv")

df_gabungan = pd.concat([df_magelang, df_semarang, df_yogyakarta], ignore_index=True)
df_gabungan

,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang
...,...,...,...,...,...,...,...
595,YOG-2195,2026-08-25,Rumah Tangga,4,250000,Kartu Kredit,Yogyakarta
596,YOG-2196,2026-08-18,Fashion,1,150000,COD,Yogyakarta
597,YOG-2197,2026-08-27,Makanan & Minuman,4,25000,Kartu Kredit,Yogyakarta
598,YOG-2198,2026-08-07,Rumah Tangga,1,25000,COD,Yogyakarta


In [12]:
df_gabungan["kota"].value_counts()

kota
Magelang      200
Semarang      200
Yogyakarta    200
Name: count, dtype: int64

#### D. Mengolah dan Download Hasil ke Direktori Processed

In [13]:
# Menambah kolom total_pendapatan
df_gabungan["total_pendapatan"] = df_gabungan["unit_terjual"] * df_gabungan["harga_satuan"]
df_gabungan.head()

,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota,total_pendapatan
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang,50000
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang,175000
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang,175000
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang,150000
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang,700000


In [14]:
ringkasan_kota_kategori = df_gabungan.groupby(["kota", "kategori"])["total_pendapatan"].sum().reset_index()
ringkasan_kota_kategori

,kota,kategori,total_pendapatan
0,Magelang,Elektronik,18775000
1,Magelang,Fashion,27750000
2,Magelang,Kesehatan & Kecantikan,17375000
3,Magelang,Makanan & Minuman,17525000
4,Magelang,Rumah Tangga,12200000
5,Semarang,Elektronik,21425000
6,Semarang,Fashion,26425000
7,Semarang,Kesehatan & Kecantikan,13525000
8,Semarang,Makanan & Minuman,19750000
9,Semarang,Rumah Tangga,10975000


In [15]:
# Menyimpan dua berkas
df_gabungan.to_csv("data_gabungan_bersih.csv", index=False)
ringkasan_kota_kategori.to_csv("ringkasan_kota_kategori.csv", index=False)
print("Kedua berkas berhasil disimpan di disk lokal.")

Kedua berkas berhasil disimpan di disk lokal.


In [16]:
# Mengunggah kedua berkas hasil olahan
!hdfs dfs -put data_gabungan_bersih.csv /user/putrisenja/ecommerce/processed
!hdfs dfs -put ringkasan_kota_kategori.csv /user/putrisenja/ecommerce/processed

# Pembuktian kedua berkas berhasil terunggah
!hdfs dfs -ls /user/putrisenja/ecommerce/processed

Found 2 items
-rw-r--r--   1 putrisenja supergroup      41257 2026-09-09 07:15 /user/putrisenja/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 putrisenja supergroup        530 2026-09-09 07:15 /user/putrisenja/ecommerce/processed/ringkasan_kota_kategori.csv


#### E. Dokumentasi & Refleksi

##### Refleksi
Apa keuntungan menyimpan data mentah (raw) terpisah dari data olahan (processed) di HDFS dibandingkan menyimpan semuanya bercampur dalam satu folder ? 

 Memisahkan data mentah dari data olahan di HDFS merupakan langkah krusial untuk menjamin keamanan serta integritas tata kelola Big Data. Praktik terbaik ini menciptakan Single Source of Truth abadi yang melindungi data asli dari risiko korup akibat kegagalan skrip pemrosesan, sehingga proses analisis ulang selalu bisa dijalankan kapan saja secara aman. Selain itu, pemisahan folder mempermudah implementasi kontrol akses berbasis peran demi melindungi informasi sensitif sekaligus memacu performa query karena mesin analitik dapat langsung memproses data olahan berformat optimal tanpa terhambat file mentah yang berantakan. Terakhir, manajemen retensi menjadi jauh lebih terstruktur melalui penerapan kebijakan penyimpanan yang berbeda untuk setiap folder sesuai siklus hidup datanya.

